# Causal-Retain: Uplift Modeling & Customer Revenue Recovery Engine
## End-to-End Walkthrough: From Latent Counterfactuals to Executive Financial ROI

### Executive Overview & Problem Formulation
In modern B2B SaaS retention marketing, standard propensity-to-churn models identify accounts with high risk of churn ($P(\text{Churn} \mid X)$) and target them with expensive retention interventions (e.g., dedicated CSM check-ins, executive renewal discounts, engineer support credits).

However, **correlation does not imply causation**. Churn propensity models suffer from two catastrophic financial failures:
1. **Wasting Budget on 'Sure Things'**: Customers who would have renewed anyway receive costly discounts, eroding margin with zero incremental benefit.
2. **Triggering Churn on 'Sleeping Dogs'**: Disgruntled or disengaged accounts that were passively auto-renewing are disturbed by outreach, causing them to reassess and cancel (negative uplift: $\tau(X) < 0$).

### The Causal AI Paradigm Shift
Causal uplift modeling shifts optimization from predicting **retention probability** to predicting the **Individual Treatment Effect (ITE)**, also known as the **Conditional Average Treatment Effect (CATE)**:

$$\tau(X_i) = \mathbb{E}[Y_i(1) - Y_i(0) \mid X_i]$$

Where:
- $Y_i(1) \in \{0, 1\}$: Potential retention outcome if account $i$ receives intervention ($W_i = 1$).
- $Y_i(0) \in \{0, 1\}$: Potential retention outcome if account $i$ receives no intervention ($W_i = 0$).

By estimating $\tau(X_i)$, we can segment accounts into the classical **Four Causal Archetypes**:
- **Persuadables** ($\tau > 0$): Only renew if contacted. **(Primary Target)**
- **Sure Things** ($\tau = 0, Y(0) = 1$): Renew regardless. **(Do Not Contact - Save Budget)**
- **Lost Causes** ($\tau = 0, Y(0) = 0$): Churn regardless. **(Do Not Contact - Save Budget)**
- **Sleeping Dogs** ($\tau < 0$): Churn *because* contacted. **(Suppress Outreach - Prevent Destruction)**


In [1]:
# 0. Environment & OpenMP Initialization
import os
import sys

# Ensure repository root is on sys.path
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
repo_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# OpenMP safety import
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Causal-Retain Modules
from data.generate_telemetry import generate_synthetic_telemetry
from src.data_pipeline import TelemetryDataPipeline, split_causal_dataset
from src.models.propensity_baseline import PropensityBaseline
from src.models.s_learner import SLearner
from src.models.t_learner import TLearner
from src.models.x_learner import XLearner
from src.evaluation.qini_metric import compute_qini_curve
from src.evaluation.decile_analysis import compute_decile_analysis
from src.explainability.uplift_shap import UpliftTreeExplainer
from src.simulation.roi_optimizer import CausalROIOptimizer

# Matplotlib non-interactive rendering for notebooks
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)

print('Environment initialized successfully. LightGBM version:', lgb.__version__)


Environment initialized successfully. LightGBM version: 4.7.0


---
## 1. Synthetic Telemetry Generation with Latent Potential Outcomes

We simulate a realistic enterprise B2B SaaS cohort with 15,000 customer accounts. The generator embeds structural latent equations defining:
- $Y(0)$: Counterfactual retention under Control.
- $Y(1)$: Counterfactual retention under Treatment.
- $\tau_{\text{true}} = Y(1) - Y(0)$: Latent ground truth Individual Treatment Effect.
- Archetype labels: `Persuadable`, `Sure Thing`, `Sleeping Dog`, `Lost Cause`.


In [2]:
# Generate 15,000 synthetic customer accounts
df_raw = generate_synthetic_telemetry(n_samples=15_000, treatment_prob=0.5, random_seed=42)

print(f'Generated {len(df_raw):,} records with {df_raw.shape[1]} columns.')
display_cols = ['user_id', 'monthly_recurring_revenue', 'contract_type', 'csat_score', 'treatment_assigned', 'retained', 'archetype', 'tau_true']
print(df_raw[display_cols].head())


Generated 15,000 records with 18 columns.
            user_id  monthly_recurring_revenue  ...     archetype  tau_true
0  usr_236e40b6bdf0                     170.18  ...    Sure Thing         0
1  usr_2fd852a0a184                     260.77  ...    Sure Thing         0
2  usr_77537bc9fced                      26.15  ...    Sure Thing         0
3  usr_d3c992b3bad8                     205.30  ...    Sure Thing         0
4  usr_23c414e7dc87                      54.22  ...  Sleeping Dog        -1

[5 rows x 8 columns]


In [3]:
# Inspect the 4 Latent Archetypes Distribution
archetype_counts = df_raw['archetype'].value_counts(normalize=True) * 100
print('Archetype Distribution (%):')
for arch, pct in archetype_counts.items():
    print(f'  - {arch:<15}: {pct:5.2f}%')

# Summary statistics by archetype
archetype_stats = df_raw.groupby('archetype')[['retained', 'treatment_assigned', 'tau_true']].mean()
print('\nArchetype Summary Stats:')
print(archetype_stats)


Archetype Distribution (%):
  - Sure Thing     : 55.63%
  - Persuadable    : 25.43%
  - Sleeping Dog   : 10.17%
  - Lost Cause     :  8.76%

Archetype Summary Stats:
              retained  treatment_assigned  tau_true
archetype                                           
Lost Cause    0.000000            0.502283       0.0
Persuadable   0.492267            0.492267       1.0
Sleeping Dog  0.492792            0.507208      -1.0
Sure Thing    1.000000            0.498981       0.0


---
## 2. Leakage-Free Preprocessing & Stratified Splitting

To ensure real-world statistical validity:
1. **Joint Stratification**: The train/test split must stratify across both treatment assignment $W$ and factual retention $Y$ simultaneously ($4$ strata: $(W=0, Y=0), (W=0, Y=1), (W=1, Y=0), (W=1, Y=1)$).
2. **Strict Information Hygiene**: Numerical scalers and categorical encoders are fit solely on the training fold, then applied out-of-sample to the test fold.
3. **Latent Isolation**: Latent fields (`tau_true`, `archetype`, `y_control_latent`, `y_treated_latent`) are stripped from feature matrices $X$ and retained strictly as held-out evaluation ground truth.


In [4]:
# Split dataset with joint (W, Y) stratification (70% train / 30% test)
split_data = split_causal_dataset(df_raw, test_size=0.30, random_state=42)

X_train, y_train, w_train = split_data.X_train, split_data.y_train, split_data.w_train
X_test, y_test, w_test = split_data.X_test, split_data.y_test, split_data.w_test

print(f'Training fold: X={X_train.shape}, Treatment rate={w_train.mean():.4f}, Retention rate={y_train.mean():.4f}')
print(f'Testing fold:  X={X_test.shape},  Treatment rate={w_test.mean():.4f}, Retention rate={y_test.mean():.4f}')
print('Engineered Features:', list(X_train.columns))


Training fold: X=(10500, 14), Treatment rate=0.4984, Retention rate=0.7316
Testing fold:  X=(4500, 14),  Treatment rate=0.4984, Retention rate=0.7318
Engineered Features: ['account_age_months', 'monthly_recurring_revenue', 'active_users_ratio', 'login_frequency_trend_30d', 'feature_usage_diversity', 'support_tickets_90d', 'unresolved_p1_tickets', 'csat_score', 'payment_failure_events', 'plan_tier_Enterprise', 'plan_tier_Professional', 'plan_tier_Starter', 'contract_type_Annual', 'contract_type_Monthly']


---
## 3. Training the Model Suite: Propensity Baseline vs Causal Meta-Learners

We fit four models:
1. **Propensity Baseline**: Flawed churn propensity model $\hat{p}(X) = P(Y=0 \mid X)$.
2. **S-Learner (Single Model)**: Estimates $\mu(X, W)$ using a single LightGBM with treatment $W$ as an explicit feature.
3. **T-Learner (Two Models)**: Estimates $\mu_0(X) = \mathbb{E}[Y \mid X, W=0]$ and $\mu_1(X) = \mathbb{E}[Y \mid X, W=1]$ with calibrated probability outputs.
4. **X-Learner (Crossover Imputation)**: Two-stage counterfactual imputation designed for unbalanced or heterogeneous response distributions.


In [5]:
# 1. Propensity Baseline (Flawed Churn Risk Model)
propensity_baseline = PropensityBaseline(random_state=42)
propensity_baseline.fit(X_train, w_train, y_train)

# 2. S-Learner (Single Model)
s_learner = SLearner(random_state=42)
s_learner.fit(X_train, w_train, y_train)

# 3. T-Learner (Dual Model with probability calibration)
t_learner = TLearner(calibrate_probabilities=True, calibration_method='sigmoid', random_state=42)
t_learner.fit(X_train, w_train, y_train)

# 4. X-Learner (Counterfactual Imputation)
x_learner = XLearner(random_state=42)
x_learner.fit(X_train, w_train, y_train)

print('All four models successfully trained on 10,500 training instances.')


All four models successfully trained on 10,500 training instances.


---
## 4. Causal Evaluation: Qini Curves, AUUC & Decile Monotonicity

Because counterfactuals are unobservable in factual production data (the Fundamental Problem of Causal Inference), traditional ROC-AUC cannot assess uplift.

We evaluate models using:
1. **Cumulative Qini Curve** $Q(u)$:
   $$Q(u) = Y_t(u) - Y_c(u) \cdot \frac{N_t}{N_c}$$
2. **Normalized Qini Score** $Q_{\text{norm}}$:
   $$Q_{\text{norm}} = \frac{\text{AUUC}_{\text{model}} - \text{AUUC}_{\text{random}}}{\text{AUUC}_{\text{optimal}} - \text{AUUC}_{\text{random}}}$$
3. **10-Decile Uplift Monotonicity**: Verifies whether incremental uplift declines monotonically from Decile 1 to Decile 10, with negative uplift isolated in the bottom deciles.


In [6]:
# Predict uplift on the held-out test fold (N=4,500)
tau_propensity = propensity_baseline.predict_uplift(X_test)
tau_s = s_learner.predict_uplift(X_test)
tau_t = t_learner.predict_uplift(X_test)
tau_x = x_learner.predict_uplift(X_test)

# Compute Qini curves and AUUC
qini_prop = compute_qini_curve(y_true=y_test, uplift_preds=tau_propensity, treatment=w_test)
qini_s = compute_qini_curve(y_true=y_test, uplift_preds=tau_s, treatment=w_test)
qini_t = compute_qini_curve(y_true=y_test, uplift_preds=tau_t, treatment=w_test)
qini_x = compute_qini_curve(y_true=y_test, uplift_preds=tau_x, treatment=w_test)

# Benchmark Scorecard
benchmark_df = pd.DataFrame([
    {'Model': 'Propensity Baseline (Flawed)', 'AUUC': qini_prop.auuc_model, 'Normalized Qini Score (Q_norm)': qini_prop.qini_score},
    {'Model': 'S-Learner (Single Model)',     'AUUC': qini_s.auuc_model,    'Normalized Qini Score (Q_norm)': qini_s.qini_score},
    {'Model': 'T-Learner (Calibrated Dual)',  'AUUC': qini_t.auuc_model,    'Normalized Qini Score (Q_norm)': qini_t.qini_score},
    {'Model': 'X-Learner (Imputed Meta)',     'AUUC': qini_x.auuc_model,    'Normalized Qini Score (Q_norm)': qini_x.qini_score},
    {'Model': 'Random Outreach Policy',       'AUUC': qini_t.auuc_random,   'Normalized Qini Score (Q_norm)': 0.0},
    {'Model': 'Theoretical Oracle Optimal',   'AUUC': qini_t.auuc_optimal,  'Normalized Qini Score (Q_norm)': 1.0},
])
print('MODEL PERFORMANCE BENCHMARK LEADERBOARD:')
print(benchmark_df.to_string(index=False))


MODEL PERFORMANCE BENCHMARK LEADERBOARD:
                       Model        AUUC  Normalized Qini Score (Q_norm)
Propensity Baseline (Flawed)  232.226295                        0.059083
    S-Learner (Single Model)  311.968488                        0.136148
 T-Learner (Calibrated Dual)  311.102403                        0.135311
    X-Learner (Imputed Meta)  308.771139                        0.133058
      Random Outreach Policy  171.090164                        0.000000
  Theoretical Oracle Optimal 1205.835665                        1.000000


In [7]:
# 10-Decile Monotonicity & Sleeping Dogs Isolation for T-Learner
decile_res = compute_decile_analysis(y_true=y_test, uplift_preds=tau_t, treatment=w_test, n_deciles=10)

print(f'Spearman Monotonicity Score: {decile_res.monotonicity_spearman_corr:.4f} (p={decile_res.monotonicity_p_value:.3e})')
print(f'Sleeping Dogs Detected in Decile 10: {decile_res.sleeping_dogs_isolated}')
print('\nDecile Analysis Table:')
print(decile_res.decile_table[['decile', 'empirical_uplift', 'rate_treated', 'rate_control', 'incremental_conversions']].to_string(index=False))


Spearman Monotonicity Score: 0.9879 (p=9.307e-08)
Sleeping Dogs Detected in Decile 10: True

Decile Analysis Table:
 decile  empirical_uplift  rate_treated  rate_control  incremental_conversions
      1          0.662721      0.857843      0.195122               298.224534
      2          0.509018      0.872247      0.363229               229.058098
      3          0.299057      0.840183      0.541126               134.575698
      4          0.122863      0.803419      0.680556                55.288462
      5          0.147622      0.850679      0.703057                66.429884
      6          0.082490      0.866972      0.784483                37.120373
      7          0.001583      0.818605      0.817021                 0.712519
      8         -0.020919      0.852321      0.873239                -9.413443
      9         -0.099126      0.713080      0.812207               -44.606882
     10         -0.221372      0.623377      0.844749               -99.617506


---
## 5. Causal Explainability: Differential TreeSHAP Attribution

Standard TreeSHAP explains a single prediction $\phi(f(X))$. In Causal ML, we compute **differential uplift attribution**:

$$\phi_i^{\text{uplift}} = \phi_i(\mu_1) - \phi_i(\mu_0)$$

This decomposes which specific account behaviors explain positive uplift (Persuadability) versus negative uplift (Sleeping Dogs).


In [8]:
# Initialize UpliftTreeExplainer on calibrated T-Learner
explainer = UpliftTreeExplainer(t_learner, feature_names=list(X_test.columns))
explanation = explainer.explain(X_test)

# Top Features Driving Differential Uplift
print('TOP UPLIFT DRIVERS (Differential TreeSHAP):')
print(explanation.feature_importance.head(10).to_string(index=False))


TOP UPLIFT DRIVERS (Differential TreeSHAP):
                  feature  mean_abs_uplift_shap  mean_uplift_shap  rank
login_frequency_trend_30d              1.164434          0.021339     1
    unresolved_p1_tickets              0.463556          0.003556     2
               csat_score              0.341821         -0.002466     3
       active_users_ratio              0.326279          0.008687     4
       account_age_months              0.125779         -0.004008     5
monthly_recurring_revenue              0.110174         -0.003950     6
  feature_usage_diversity              0.108641         -0.002699     7
     contract_type_Annual              0.079437         -0.002069     8
   payment_failure_events              0.068455         -0.012219     9
      support_tickets_90d              0.043098         -0.000015    10


In [9]:
# Archetype Deep Dive: Persuadables vs Sleeping Dogs
arch_summaries = explainer.explain_archetypes(explanation, split_data.latent_test['archetype'].values)
print('ARCHETYPE DIFFERENTIATION - Top Features for Persuadables:')
print(arch_summaries['Persuadable'].head(5).to_string(index=False))
print('\nARCHETYPE DIFFERENTIATION - Top Features for Sleeping Dogs:')
print(arch_summaries['Sleeping Dog'].head(5).to_string(index=False))


ARCHETYPE DIFFERENTIATION - Top Features for Persuadables:
                  feature  mean_uplift_shap  mean_abs_uplift_shap
login_frequency_trend_30d          0.658406              1.206363
    unresolved_p1_tickets          0.303829              0.697156
       active_users_ratio          0.088067              0.325489
               csat_score          0.089770              0.284079
       account_age_months          0.005006              0.119356

ARCHETYPE DIFFERENTIATION - Top Features for Sleeping Dogs:
                  feature  mean_uplift_shap  mean_abs_uplift_shap
login_frequency_trend_30d         -0.901215              1.259088
               csat_score         -0.144835              0.456008
    unresolved_p1_tickets         -0.149917              0.339612
       active_users_ratio         -0.027800              0.323792
       account_age_months         -0.001928              0.131865


---
## 6. Financial Optimization: Executive ROI & Revenue Recovery

We formulate the **Causal Net Profit Optimization Problem**:

$$\max_k \Delta \text{Profit}(k) = \sum_{i=1}^k (\hat{\tau}_i \cdot \text{ARR}_i) - k \cdot C_{\text{intervention}}$$

Subject to:
$$k \cdot C_{\text{intervention}} \le B_{\text{campaign}}$$

The marginal profit inflection point $k^*$ occurs exactly where:
$$\hat{\tau}_i \cdot \text{ARR}_i \ge C_{\text{intervention}}$$


In [10]:
# Execute Head-to-Head ROI Strategy Comparison
INTERVENTION_COST = 50.0   # $50 cost per customer touchpoint
CAMPAIGN_BUDGET = 150_000.0  # $150,000 campaign budget cap

optimizer = CausalROIOptimizer(intervention_cost=INTERVENTION_COST, campaign_budget=CAMPAIGN_BUDGET)

head_to_head = optimizer.compare_targeting_strategies(
    uplift_preds=tau_t,
    churn_propensity=tau_propensity,
    mrr=split_data.mrr_test,
    archetypes=split_data.latent_test['archetype'].values,
)

print('=' * 65)
print('EXECUTIVE REVENUE RECOVERY SCORECARD')
print('=' * 65)
print(f'Cohort Size:                       {len(y_test):,} customer accounts')
print(f'Causal Optimal Cutoff (k*):        {head_to_head.causal_policy.n_targeted:,} accounts ({head_to_head.causal_policy.fraction_targeted*100:.1f}%)')
print(f'Propensity Optimal Cutoff (k*):    {head_to_head.propensity_policy.n_targeted:,} accounts ({head_to_head.propensity_policy.fraction_targeted*100:.1f}%)')
print('-' * 65)
print(f'Causal Strategy Net Profit:        ${head_to_head.causal_policy.net_profit_delta:,.2f}')
print(f'Propensity Strategy Net Profit:    ${head_to_head.propensity_policy.net_profit_delta:,.2f}')
print(f'Net Advantage of Causal AI:        +${head_to_head.net_profit_advantage:,.2f} ({head_to_head.efficiency_multiplier:.2f}x ROI multiplier)')
print('-' * 65)
print(f'Sure Things Waste Avoided:         ${head_to_head.sure_things_waste_avoided:,.2f}')
print(f'Sleeping Dogs ARR Protected:       ${head_to_head.sleeping_dogs_destruction_prevented:,.2f}')
print(f'Net Dollar Retention (NDR) Lift:   +{head_to_head.causal_policy.ndr_delta_percentage:.2f}%')
print('=' * 65)


EXECUTIVE REVENUE RECOVERY SCORECARD
Cohort Size:                       4,500 customer accounts
Causal Optimal Cutoff (k*):        2,735 accounts (60.8%)
Propensity Optimal Cutoff (k*):    2,735 accounts (60.8%)
-----------------------------------------------------------------
Causal Strategy Net Profit:        $2,104,686.51
Propensity Strategy Net Profit:    $1,326,225.16
Net Advantage of Causal AI:        +$778,461.35 (1.59x ROI multiplier)
-----------------------------------------------------------------
Sure Things Waste Avoided:         $52,200.00
Sleeping Dogs ARR Protected:       $729,010.56
Net Dollar Retention (NDR) Lift:   +15.69%


---
## 7. Strategic Conclusions & Production Architecture Recommendations

### Key Empirical Findings:
1. **2.29x Targeting Efficiency**: The calibrated T-Learner achieved a Normalized Qini Score of **0.135** (AUUC 311.1), outperforming the standard Propensity-to-churn baseline (**0.059**, AUUC 232.2) by **2.29x**.
2. **Substantial EBITDA Uplift**: Optimizing the targeting cutoff using marginal profitability generated substantial incremental net profit over traditional propensity scoring on the test cohort.
3. **Elimination of Budget Waste**: Causal AI avoided spending retention marketing dollars on **Sure Things** (accounts that renew regardless).
4. **Active Revenue Protection**: Standard propensity models targeted **Sleeping Dogs** because their churn propensity was high, causing severe cancellation spikes. Causal uplift modeling successfully identified and suppressed outreach to this group, protecting substantial ARR.

### Production Deployment Architecture:
- **Streaming & Batch Pipeline**: Model artifacts serialized via standard Scikit-Learn pipelines. Telemetry processed daily or weekly via Snowflake / BigQuery.
- **Microservice / Front-End**: Interactive executive decision dashboard running on Streamlit with sub-second parameter optimization.
- **Continuous A/B Holdouts**: A persistent 5% random holdout group must be maintained in production campaigns to continuously validate counterfactual estimates against observed factual uplift.
